下面是在上一版 TrainMonitor 基础上扩展的带自动干预版本。核心思路是：监控器检测到异常后，不再只是打印告警，而是通过回调函数直接修改训练状态（学习率、KL 系数、跳过当前 step 等），在崩溃前自动止损。

## 自动干预策略一览

| 异常信号 | 干预动作 | 恢复机制 |
|----------|----------|----------|
| 梯度范数 > 爆炸阈值 | 学习率 × 0.5，跳过当前 step | 连续 200 步健康后恢复原 LR |
| KL 散度持续偏高 | KL 系数 × 1.5 | KL 回落到正常区间后逐步衰减 |
| 策略熵坍塌 | 增大熵正则系数 | 熵恢复后逐步衰减 |
| 检测到 NaN | 立即跳过 step，不更新参数 | 连续 NaN 触发紧急保存并停止 |
| ratio 溢出 | 降低学习率，增大裁剪范围 | 稳定后恢复 |

## 完整代码

In [ ]:
import numpy as np
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Dict, Optional, Callable
from enum import Enum


class AlertLevel(Enum):
    OK = "ok"
    WARNING = "warning"
    CRITICAL = "critical"


@dataclass
class ThresholdConfig:
    """各指标的健康/警告/危险阈值"""
    # 奖励分布
    reward_std_min: float = 0.1
    reward_range_ratio_min: float = 0.1
    reward_tail_ratio_max: float = 5.0
    zero_var_group_ratio_max: float = 0.3

    # 概率比 ratio
    ratio_p50_low: float = 0.9
    ratio_p50_high: float = 1.1
    ratio_p99_max: float = 10.0
    ratio_p1_min: float = 0.1

    # 优势值
    advantage_std_min: float = 0.05
    advantage_p99_max: float = 10.0

    # KL 散度
    kl_mean_max: float = 0.05
    kl_p95_warning: float = 0.03
    kl_p95_critical: float = 0.08

    # 梯度范数
    grad_norm_warning: float = 5.0
    grad_norm_critical: float = 10.0

    # 策略熵
    entropy_ratio_min: float = 0.2

    # NaN
    max_consecutive_nan_steps: int = 3


@dataclass
class InterventionConfig:
    """自动干预参数"""
    # 学习率干预
    lr_decay_factor: float = 0.5            # 每次干预时 LR 乘以该系数
    lr_min: float = 1e-7                    # LR 下限，不再继续衰减
    lr_recovery_steps: int = 200            # 连续健康多少步后尝试恢复 LR
    lr_recovery_factor: float = 1.2         # 恢复时 LR 乘以该系数

    # KL 系数干预
    kl_coef_increase_factor: float = 1.5    # KL 超标时系数乘以该值
    kl_coef_max: float = 1.0                # KL 系数上限
    kl_coef_decay_factor: float = 0.95      # KL 正常后逐步衰减
    kl_coef_min: float = 0.001              # KL 系数下限

    # 熵系数干预
    entropy_coef_increase_factor: float = 2.0
    entropy_coef_max: float = 0.1
    entropy_coef_decay_factor: float = 0.98
    entropy_coef_min: float = 0.001

    # 紧急停止
    emergency_stop_after_nan: int = 3       # 连续 NaN 超过此数则紧急停止


@dataclass
class InterventionResult:
    """单次干预的结果记录"""
    step: int
    trigger: str
    action: str
    old_value: float
    new_value: float
    level: AlertLevel


class AutoInterventionMonitor:
    """
    带自动干预的训练监控器。

    用法：
        monitor = AutoInterventionMonitor(
            log_every=50,
            warmup_steps=200,
            initial_lr=1e-5,
            initial_kl_coef=0.01,
            initial_entropy_coef=0.01,
        )

        for step in range(total_steps):
            # 1. 检查是否应该跳过当前 step
            if monitor.should_skip_step(step):
                print(f"[Step {step}] 跳过（干预生效中）")
                continue

            # 2. 执行训练
            loss, batch_data = train_step(...)

            # 3. 上报指标，获取干预结果
            interventions = monitor.update(step=step, metrics={
                "rewards": batch_data["rewards"],
                "ratios": batch_data["ratios"],
                "advantages": batch_data["advantages"],
                "kl": batch_data["kl"],
                "grad_norm": compute_grad_norm(model),
                "policy_entropy": compute_entropy(model),
                "has_nan": check_nan(model),
            })

            # 4. 应用干预（修改优化器参数等）
            for iv in interventions:
                print(f"  ⚡ 干预: {iv.trigger} → {iv.action} "
                      f"({iv.old_value:.6f} → {iv.new_value:.6f})")

            # 5. 检查是否需要紧急停止
            if monitor.should_emergency_stop():
                save_checkpoint(model, f"emergency_step_{step}.pt")
                print("🛑 紧急停止！")
                break

            # 6. 正常更新参数
            optimizer.step()
    """

    def __init__(
        self,
        log_every: int = 50,
        warmup_steps: int = 200,
        initial_lr: float = 1e-5,
        initial_kl_coef: float = 0.01,
        initial_entropy_coef: float = 0.01,
        thresholds: Optional[ThresholdConfig] = None,
        intervention_config: Optional[InterventionConfig] = None,
        logger=None,
    ):
        self.log_every = log_every
        self.warmup_steps = warmup_steps
        self.thresholds = thresholds or ThresholdConfig()
        self.ic = intervention_config or InterventionConfig()
        self.logger = logger

        # --- 可干预状态 ---
        self.current_lr = initial_lr
        self.original_lr = initial_lr
        self.current_kl_coef = initial_kl_coef
        self.original_kl_coef = initial_kl_coef
        self.current_entropy_coef = initial_entropy_coef
        self.original_entropy_coef = initial_entropy_coef

        # --- 内部追踪 ---
        self._history = defaultdict(lambda: deque(maxlen=1000))
        self._baseline = {}
        self._initial_entropy = None
        self._consecutive_nan_steps = 0
        self._consecutive_healthy_steps = 0
        self._skip_current_step = False
        self._emergency_stop = False
        self._interventions_log: list[InterventionResult] = []
        self._step_count = 0

        # 冷却计数器：防止干预过于频繁
        self._lr_cooldown = 0
        self._kl_coef_cooldown = 0
        self._entropy_coef_cooldown = 0

    # ================================================================
    #  公开接口
    # ================================================================

    def should_skip_step(self, step: int) -> bool:
        """判断当前 step 是否应被跳过（上一步触发了严重干预）"""
        return self._skip_current_step

    def should_emergency_stop(self) -> bool:
        """判断是否应紧急停止训练"""
        return self._emergency_stop

    def get_current_lr(self) -> float:
        return self.current_lr

    def get_current_kl_coef(self) -> float:
        return self.current_kl_coef

    def get_current_entropy_coef(self) -> float:
        return self.current_entropy_coef

    def update(self, step: int, metrics: Dict) -> list[InterventionResult]:
        """
        每步调用。返回本轮触发的干预列表。
        调用方应根据返回值修改优化器 / 损失函数中的对应系数。
        """
        self._step_count = step
        self._skip_current_step = False
        interventions = []

        # 1. 计算统计量
        stats = self._compute_stats(metrics)
        for k, v in stats.items():
            self._history[k].append(v)

        # 2. warmup 阶段：只记录，不干预
        if step <= self.warmup_steps:
            if step == self.warmup_steps:
                self._build_baseline()
                print(f"[Monitor] 基线建立完成 (warmup={self.warmup_steps} 步)")
            return interventions

        # 3. 记录初始熵
        if self._initial_entropy is None and "policy_entropy" in stats:
            self._initial_entropy = stats["policy_entropy"]

        # 4. NaN 检测（最高优先级）
        if stats.get("has_nan", False):
            interventions.extend(self._handle_nan(step))
        else:
            self._consecutive_nan_steps = 0

        # 5. 梯度范数干预
        if "grad_norm" in stats:
            interventions.extend(self._handle_grad_norm(step, stats))

        # 6. KL 散度干预
        if "kl_p95" in stats:
            interventions.extend(self._handle_kl(step, stats))

        # 7. 策略熵干预
        if "policy_entropy" in stats and self._initial_entropy is not None:
            interventions.extend(self._handle_entropy(step, stats))

        # 8. ratio 溢出干预
        if "ratio_p99" in stats:
            interventions.extend(self._handle_ratio_overflow(step, stats))

        # 9. 恢复机制：连续健康时逐步恢复被干预的参数
        if not interventions:
            self._consecutive_healthy_steps += 1
            interventions.extend(self._try_recover(step))
        else:
            self._consecutive_healthy_steps = 0

        # 10. 冷却计数器递减
        self._lr_cooldown = max(0, self._lr_cooldown - 1)
        self._kl_coef_cooldown = max(0, self._kl_coef_cooldown - 1)
        self._entropy_coef_cooldown = max(0, self._entropy_coef_cooldown - 1)

        # 11. 记录干预日志
        self._interventions_log.extend(interventions)

        # 12. 定期输出报告
        if step % self.log_every == 0:
            self._log_report(step, stats, interventions)

        # 13. 写入外部 logger
        if self.logger is not None:
            log_data = {**stats, "monitor/lr": self.current_lr,
                        "monitor/kl_coef": self.current_kl_coef,
                        "monitor/entropy_coef": self.current_entropy_coef}
            self.logger.log(log_data, step=step)

        return interventions

    # ================================================================
    #  干预处理器
    # ================================================================

    def _handle_nan(self, step: int) -> list[InterventionResult]:
        """NaN 检测：跳过当前 step，连续 NaN 触发紧急停止"""
        self._consecutive_nan_steps += 1
        self._skip_current_step = True
        results = []

        results.append(InterventionResult(
            step=step,
            trigger="NaN 检测",
            action="跳过当前 step（不更新参数）",
            old_value=0, new_value=0,
            level=AlertLevel.CRITICAL,
        ))

        if self._consecutive_nan_steps >= self.ic.emergency_stop_after_nan:
            self._emergency_stop = True
            results.append(InterventionResult(
                step=step,
                trigger=f"连续 {self._consecutive_nan_steps} 步 NaN",
                action="紧急停止训练",
                old_value=0, new_value=0,
                level=AlertLevel.CRITICAL,
            ))

        return results

    def _handle_grad_norm(self, step: int, stats: Dict) -> list[InterventionResult]:
        """梯度范数超标：降低学习率"""
        gn = stats["grad_norm"]
        results = []

        if gn > self.thresholds.grad_norm_critical:
            # 严重：跳过 step + 降低 LR
            self._skip_current_step = True
            old_lr = self.current_lr
            new_lr = max(self.current_lr * self.ic.lr_decay_factor, self.ic.lr_min)
            if new_lr < old_lr:
                self.current_lr = new_lr
                self._lr_cooldown = 50  # 50 步冷却期
                results.append(InterventionResult(
                    step=step,
                    trigger=f"梯度范数 {gn:.1f} > {self.thresholds.grad_norm_critical}",
                    action=f"LR 降低 ×{self.ic.lr_decay_factor} + 跳过 step",
                    old_value=old_lr, new_value=new_lr,
                    level=AlertLevel.CRITICAL,
                ))

        elif gn > self.thresholds.grad_norm_warning and self._lr_cooldown == 0:
            # 警告：降低 LR
            old_lr = self.current_lr
            new_lr = max(self.current_lr * self.ic.lr_decay_factor, self.ic.lr_min)
            if new_lr < old_lr:
                self.current_lr = new_lr
                self._lr_cooldown = 30
                results.append(InterventionResult(
                    step=step,
                    trigger=f"梯度范数 {gn:.1f} > {self.thresholds.grad_norm_warning}",
                    action=f"LR 降低 ×{self.ic.lr_decay_factor}",
                    old_value=old_lr, new_value=new_lr,
                    level=AlertLevel.WARNING,
                ))

        return results

    def _handle_kl(self, step: int, stats: Dict) -> list[InterventionResult]:
        """KL 散度过大：增大 KL 系数"""
        kl_p95 = stats["kl_p95"]
        results = []

        if kl_p95 > self.thresholds.kl_p95_critical:
            old_coef = self.current_kl_coef
            new_coef = min(
                self.current_kl_coef * self.ic.kl_coef_increase_factor,
                self.ic.kl_coef_max,
            )
            if new_coef > old_coef:
                self.current_kl_coef = new_coef
                self._kl_coef_cooldown = 30
                results.append(InterventionResult(
                    step=step,
                    trigger=f"KL p95={kl_p95:.4f} > {self.thresholds.kl_p95_critical}",
                    action=f"KL 系数增大 ×{self.ic.kl_coef_increase_factor}",
                    old_value=old_coef, new_value=new_coef,
                    level=AlertLevel.CRITICAL,
                ))

        elif kl_p95 > self.thresholds.kl_p95_warning and self._kl_coef_cooldown == 0:
            old_coef = self.current_kl_coef
            new_coef = min(
                self.current_kl_coef * self.ic.kl_coef_increase_factor,
                self.ic.kl_coef_max,
            )
            if new_coef > old_coef:
                self.current_kl_coef = new_coef
                self._kl_coef_cooldown = 20
                results.append(InterventionResult(
                    step=step,
                    trigger=f"KL p95={kl_p95:.4f} > {self.thresholds.kl_p95_warning}",
                    action=f"KL 系数增大 ×{self.ic.kl_coef_increase_factor}",
                    old_value=old_coef, new_value=new_coef,
                    level=AlertLevel.WARNING,
                ))

        return results

    def _handle_entropy(self, step: int, stats: Dict) -> list[InterventionResult]:
        """策略熵坍塌：增大熵正则系数"""
        entropy_ratio = stats["policy_entropy"] / self._initial_entropy
        results = []

        if entropy_ratio < self.thresholds.entropy_ratio_min:
            old_coef = self.current_entropy_coef
            new_coef = min(
                self.current_entropy_coef * self.ic.entropy_coef_increase_factor,
                self.ic.entropy_coef_max,
            )
            if new_coef > old_coef and self._entropy_coef_cooldown == 0:
                self.current_entropy_coef = new_coef
                self._entropy_coef_cooldown = 50
                results.append(InterventionResult(
                    step=step,
                    trigger=f"策略熵仅为初始值的 {entropy_ratio:.0%}",
                    action=f"熵系数增大 ×{self.ic.entropy_coef_increase_factor}",
                    old_value=old_coef, new_value=new_coef,
                    level=AlertLevel.CRITICAL,
                ))

        return results

    def _handle_ratio_overflow(self, step: int, stats: Dict) -> list[InterventionResult]:
        """ratio 溢出：降低学习率"""
        ratio_p99 = stats["ratio_p99"]
        results = []

        if ratio_p99 > self.thresholds.ratio_p99_max and self._lr_cooldown == 0:
            old_lr = self.current_lr
            new_lr = max(self.current_lr * self.ic.lr_decay_factor, self.ic.lr_min)
            if new_lr < old_lr:
                self.current_lr = new_lr
                self._lr_cooldown = 30
                results.append(InterventionResult(
                    step=step,
                    trigger=f"ratio p99={ratio_p99:.1f} 溢出",
                    action=f"LR 降低 ×{self.ic.lr_decay_factor}",
                    old_value=old_lr, new_value=new_lr,
                    level=AlertLevel.WARNING,
                ))

        return results

    def _try_recover(self, step: int) -> list[InterventionResult]:
        """连续健康时逐步恢复被干预的参数"""
        results = []

        if self._consecutive_healthy_steps < self.ic.lr_recovery_steps:
            return results

        # 恢复 LR
        if self.current_lr < self.original_lr:
            old_lr = self.current_lr
            new_lr = min(self.current_lr * self.ic.lr_recovery_factor, self.original_lr)
            if new_lr > old_lr:
                self.current_lr = new_lr
                results.append(InterventionResult(
                    step=step,
                    trigger=f"连续 {self._consecutive_healthy_steps} 步健康",
                    action=f"LR 恢复 ×{self.ic.lr_recovery_factor}",
                    old_value=old_lr, new_value=new_lr,
                    level=AlertLevel.OK,
                ))

        # 恢复 KL 系数
        if self.current_kl_coef > self.original_kl_coef:
            old_coef = self.current_kl_coef
            new_coef = max(
                self.current_kl_coef * self.ic.kl_coef_decay_factor,
                self.original_kl_coef,
            )
            if new_coef < old_coef:
                self.current_kl_coef = new_coef
                results.append(InterventionResult(
                    step=step,
                    trigger=f"连续 {self._consecutive_healthy_steps} 步健康",
                    action=f"KL 系数衰减 ×{self.ic.kl_coef_decay_factor}",
                    old_value=old_coef, new_value=new_coef,
                    level=AlertLevel.OK,
                ))

        # 恢复熵系数
        if self.current_entropy_coef > self.original_entropy_coef:
            old_coef = self.current_entropy_coef
            new_coef = max(
                self.current_entropy_coef * self.ic.entropy_coef_decay_factor,
                self.original_entropy_coef,
            )
            if new_coef < old_coef:
                self.current_entropy_coef = new_coef
                results.append(InterventionResult(
                    step=step,
                    trigger=f"连续 {self._consecutive_healthy_steps} 步健康",
                    action=f"熵系数衰减 ×{self.ic.entropy_coef_decay_factor}",
                    old_value=old_coef, new_value=new_coef,
                    level=AlertLevel.OK,
                ))

        return results

    # ================================================================
    #  统计计算
    # ================================================================

    def _compute_stats(self, metrics: Dict) -> Dict:
        stats = {}

        if "rewards" in metrics:
            r = np.asarray(metrics["rewards"], dtype=np.float64)
            stats["reward_mean"] = float(np.mean(r))
            stats["reward_std"] = float(np.std(r))
            stats["reward_p5"] = float(np.percentile(r, 5))
            stats["reward_p50"] = float(np.percentile(r, 50))
            stats["reward_p95"] = float(np.percentile(r, 95))
            stats["reward_p99"] = float(np.percentile(r, 99))
            r_range = stats["reward_p95"] - stats["reward_p5"]
            r_full = (r.max() - r.min()) if r.max() != r.min() else 1.0
            stats["reward_range_ratio"] = r_range / r_full

        if "ratios" in metrics:
            ratio = np.asarray(metrics["ratios"], dtype=np.float64)
            stats["ratio_p1"] = float(np.percentile(ratio, 1))
            stats["ratio_p50"] = float(np.percentile(ratio, 50))
            stats["ratio_p95"] = float(np.percentile(ratio, 95))
            stats["ratio_p99"] = float(np.percentile(ratio, 99))

        if "advantages" in metrics:
            adv = np.asarray(metrics["advantages"], dtype=np.float64)
            stats["advantage_mean"] = float(np.mean(adv))
            stats["advantage_std"] = float(np.std(adv))
            stats["advantage_p99"] = float(np.percentile(adv, 9

